In [1]:
import psycopg2
from psycopg2 import Error

def create_db_connection(host_name, port_name, user_name, user_password, dbname):
    connection = None
    try:
        connection = psycopg2.connect(dbname=dbname, user=user_name, password=user_password, host=host_name, port=port_name)
        print("openGauss Database connection successful")
    except Error as err:
        print(f"Error: {err}")
    return connection

In [3]:
connection = create_db_connection("localhost", "5432", "myuser", "123456abc.", "mydb")

openGauss Database connection successful


In [4]:
def read_query(connection, query):
    cursor = connection.cursor()
    result = None
    try:
        cursor.execute(query)
        result = cursor.fetchall()
        return result
    except Error as err:
        print(f"Error: '{err}'")

In [5]:
def execute_query(connection, query):
    cursor = connection.cursor()
    try:
        cursor.execute(query)
        connection.commit()
        print("Query executed successfully")
    except Error as err:
        print(f"Error: {err}")

In [5]:
cursor = connection.cursor()
with open('employ.sql', 'r') as file:
    sql_script = file.read()

sql_script = "DROP SCHEMA IF EXISTS kk CASCADE;\n" + sql_script

sql_commands = sql_script.split(';')
for command in sql_commands:
    if command.strip():
        cursor.execute(command)

print("employ.sql 执行成功！")
connection.commit()
connection.close()

employ.sql 执行成功！


In [7]:
#(1)
connection = create_db_connection("localhost", "5432", "myuser", "123456abc.", "mydb")

query = """
CREATE OR REPLACE PROCEDURE kk.proc_product1(
    IN p_emp_no INT,
    INOUT p_result REFCURSOR
)
AS 
DECLARE
    v_emp_info RECORD;
BEGIN
    IF NOT EXISTS (SELECT 1 FROM kk.employees WHERE emp_no = p_emp_no) THEN
        RAISE EXCEPTION 'person % not exist', p_emp_no;
    END IF;

    SELECT first_name, last_name, birth_date, gender 
    INTO v_emp_info
    FROM kk.employees 
    WHERE emp_no = p_emp_no;

    OPEN p_result FOR 
        SELECT 
            v_emp_info.first_name,
            v_emp_info.last_name,
            v_emp_info.birth_date,
            v_emp_info.gender,
            s.salary,
            s.from_date,
            s.to_date
        FROM kk.salaries s
        WHERE s.emp_no = p_emp_no;
END;
"""

execute_query(connection, query)

results = read_query(connection, """
BEGIN;
CALL kk.proc_product1(11111, 'p_result');
FETCH ALL FROM p_result;
""")

if results:
    print("-" * 100)
    for result in results:
        print(result)
else:
    print("未找到该员工的记录")

openGauss Database connection successful
Query executed successfully
----------------------------------------------------------------------------------------------------
('Udi', 'Zastre', datetime.datetime(1954, 7, 14, 0, 0), 'F', 49777, datetime.datetime(1994, 12, 25, 0, 0), datetime.datetime(1995, 12, 25, 0, 0))
('Udi', 'Zastre', datetime.datetime(1954, 7, 14, 0, 0), 'F', 52538, datetime.datetime(1995, 12, 25, 0, 0), datetime.datetime(1996, 12, 24, 0, 0))
('Udi', 'Zastre', datetime.datetime(1954, 7, 14, 0, 0), 'F', 52908, datetime.datetime(1996, 12, 24, 0, 0), datetime.datetime(1997, 12, 24, 0, 0))
('Udi', 'Zastre', datetime.datetime(1954, 7, 14, 0, 0), 'F', 52872, datetime.datetime(1997, 12, 24, 0, 0), datetime.datetime(1998, 12, 24, 0, 0))
('Udi', 'Zastre', datetime.datetime(1954, 7, 14, 0, 0), 'F', 55066, datetime.datetime(1998, 12, 24, 0, 0), datetime.datetime(1999, 12, 24, 0, 0))
('Udi', 'Zastre', datetime.datetime(1954, 7, 14, 0, 0), 'F', 59393, datetime.datetime(1999, 12, 24, 

In [9]:
#2) 创建并执行一个带输入参数和多个输出参数的存储过程proc_product2，通过该存储过程可以根据输入参数员工编号查询出该员工具体信息：包括名字、生日、性别、在不同时间段下的工资（数组类型）。【使用多个输出型变量】
connection = create_db_connection("localhost", "5432", "myuser", "123456abc.", "mydb")

query = """
CREATE OR REPLACE FUNCTION kk.proc_product2(
    IN p_emp_no INT,                     
    OUT o_first_name VARCHAR(50),        
    OUT o_last_name VARCHAR(50),         
    OUT o_birth_date DATE,              
    OUT o_gender CHAR(1),               
    OUT o_salaries JSON                
) 
AS $$
BEGIN
    SELECT first_name, last_name, birth_date, gender 
    INTO o_first_name, o_last_name, o_birth_date, o_gender
    FROM kk.employees 
    WHERE emp_no = p_emp_no;
    
    SELECT json_agg(
        json_build_object(
            'salary', salary,
            'from_date', from_date,
            'to_date', to_date
        )
    ) INTO o_salaries
    FROM kk.salaries
    WHERE emp_no = p_emp_no;
    
    IF o_first_name IS NULL THEN
        RAISE NOTICE 'person % not exist', p_emp_no;
    END IF;
END;
$$ LANGUAGE PLPGSQL;

SELECT * FROM kk.proc_product2(11111);
"""

results = read_query(connection, query)
if results:
    print("-" * 80)
    for result in results:
        print(result)
else:
    print("No results found.")

openGauss Database connection successful
--------------------------------------------------------------------------------
('Udi', 'Zastre', datetime.datetime(1954, 7, 14, 0, 0), 'F', [{'salary': 49777, 'from_date': '1994-12-25 00:00:00', 'to_date': '1995-12-25 00:00:00'}, {'salary': 52538, 'from_date': '1995-12-25 00:00:00', 'to_date': '1996-12-24 00:00:00'}, {'salary': 52908, 'from_date': '1996-12-24 00:00:00', 'to_date': '1997-12-24 00:00:00'}, {'salary': 52872, 'from_date': '1997-12-24 00:00:00', 'to_date': '1998-12-24 00:00:00'}, {'salary': 55066, 'from_date': '1998-12-24 00:00:00', 'to_date': '1999-12-24 00:00:00'}, {'salary': 59393, 'from_date': '1999-12-24 00:00:00', 'to_date': '2000-12-23 00:00:00'}, {'salary': 60720, 'from_date': '2000-12-23 00:00:00', 'to_date': '2001-12-23 00:00:00'}, {'salary': 61163, 'from_date': '2001-12-23 00:00:00', 'to_date': '9999-01-01 00:00:00'}])


In [14]:
#3) 删除存储过程 proc_product2。
query = """
DROP FUNCTION IF EXISTS kk.proc_product2(
    INT, 
    VARCHAR(50), 
    VARCHAR(50), 
    DATE, 
    CHAR(1), 
    JSON
);
"""

results = read_query(connection, query)
if results is None:
    print("Procedure dropped successfully or did not exist")
else:
    print("Drop operation returned unexpected results")

Error: 'no results to fetch'
Procedure dropped successfully or did not exist


In [16]:
#1) 创建一个函数，实现根据输入的 dept_no(员工编号)查看该员工所属部门(department)、职位(title)和工资(salary)。
connection = create_db_connection("localhost", "5432", "myuser", "123456abc.", "mydb")

query = """
CREATE OR REPLACE FUNCTION kk.func_emp_dept_title_salary(
    p_emp_no INT
)
RETURNS TABLE (
    dept_name VARCHAR(40),
    title VARCHAR(50), 
    salary INT  
) 
AS $$
BEGIN
    RETURN QUERY
    SELECT 
        d.dept_name,
        t.title,
        s.salary
    FROM 
        kk.employees e
    JOIN kk.dept_emp de ON e.emp_no = de.emp_no
    JOIN kk.departments d ON de.dept_no = d.dept_no
    JOIN kk.titles t ON e.emp_no = t.emp_no
    JOIN kk.salaries s ON e.emp_no = s.emp_no
    WHERE 
        e.emp_no = p_emp_no
        AND de.to_date = '9999-01-01' 
        AND t.to_date = '9999-01-01' 
        AND s.to_date = '9999-01-01';
END;
$$ LANGUAGE PLPGSQL;

SELECT * FROM kk.func_emp_dept_title_salary(11111);
"""

results = read_query(connection, query)
if results:
    print("dept_name | title | Salary")
    print("-" * 80)
    for result in results:
        print(result)
else:
    print("No results found.")

openGauss Database connection successful
dept_name | title | Salary
--------------------------------------------------------------------------------
('Production', 'Assistant Engineer', 61163)


In [64]:
#2) 创建一个函数 INSERT_Func，向 departments 表中插入数据（部门编号、部门名称），并将 departments 中的部门数作为返回值。
connection = create_db_connection("localhost", "5432", "myuser", "123456abc.", "mydb")

query = """
CREATE OR REPLACE FUNCTION kk.INSERT_Func(
    p_dept_no CHAR(4),   
    p_dept_name VARCHAR(40) 
)
RETURNS INTEGER        
AS $$
DECLARE
    dept_count INTEGER; 
BEGIN
    INSERT INTO kk.departments(dept_no, dept_name)
    VALUES (p_dept_no, p_dept_name);
    
    SELECT COUNT(*) INTO dept_count
    FROM kk.departments;
    
    RETURN dept_count;
END;
$$ LANGUAGE PLPGSQL;

SELECT kk.INSERT_Func('d010', 'Policy') AS result;
"""

result = read_query(connection, query)
if result:
    print("-" * 80)
    print("Function executed successfully. Result:")
    print(result[0][0])
else:
    print("No results returned or error occurred")

openGauss Database connection successful
--------------------------------------------------------------------------------
Function executed successfully. Result:
10


In [75]:
#3) 创建一个触发器函数，实现向 departments 表中插入数据。
connection = create_db_connection("localhost", "5432", "myuser", "123456abc.", "mydb")

query = """
DROP TRIGGER IF EXISTS TRI_DEPT_INSERT ON kk.departments;
CREATE OR REPLACE FUNCTION tri_dept_insert_func()
RETURNS TRIGGER
AS $$
BEGIN
    IF length(NEW.dept_name) < 3 THEN
        RAISE EXCEPTION 'The department name must contain at least 3 characters';
    END IF;
    
    IF NOT (NEW.dept_no ~ '^d\d{3}$') THEN
        RAISE EXCEPTION 'The department number format is incorrect, it should start with d followed by 3 digits (such as d001)';
    END IF;
    
    INSERT INTO kk.dept_creation_log(
        dept_no, 
        dept_name, 
        created_by, 
        creation_time
    ) VALUES (
        NEW.dept_no,
        NEW.dept_name,
        current_user,
        current_timestamp
    );
    
    RETURN NEW;
END;
$$ LANGUAGE plpgsql;

CREATE TRIGGER TRI_DEPT_INSERT
BEFORE INSERT ON kk.departments
FOR EACH ROW 
EXECUTE PROCEDURE tri_dept_insert_func();
"""

execute_query(connection, query)
read_query(connection, "INSERT INTO kk.departments VALUES ('x010', 'drdbtn');")

openGauss Database connection successful
Query executed successfully
Error: The department number format is incorrect, it should start with d followed by 3 digits (such as d001)



In [77]:
connection = create_db_connection("localhost", "5432", "myuser", "123456abc.", "mydb")

read_query(connection, "INSERT INTO kk.departments VALUES ('d011', 'A');")

openGauss Database connection successful
Error: 'The department name must contain at least 3 characters
'


In [80]:
#4) 创建并执行一个函数 FUNC1，通过该函数可以根据输入参数员工编号查询出该员工具体信息：包括名字、生日、性别、在不同时间段下的工资（放在数组中）。
connection = create_db_connection("localhost", "5432", "myuser", "123456abc.", "mydb")

query = """
CREATE OR REPLACE FUNCTION kk.FUNC1(
    p_emp_no INTEGER
)
RETURNS TABLE (
    first_name VARCHAR(50),
    last_name VARCHAR(50),
    birth_date DATE,
    gender VARCHAR(6),
    salaries JSON
) 
AS $$
BEGIN
    RETURN QUERY
    SELECT 
        e.first_name,
        e.last_name,
        e.birth_date,
        e.gender,
        (SELECT json_agg(
            json_build_object(
                'from_date', s.from_date,
                'to_date', s.to_date,
                'salary', s.salary
            )
            ORDER BY s.from_date DESC
        )
        FROM kk.salaries s
        WHERE s.emp_no = e.emp_no) AS salaries
    FROM 
        kk.employees e
    WHERE 
        e.emp_no = p_emp_no;
END;
$$ LANGUAGE PLPGSQL;

SELECT * FROM kk.FUNC1(11111);
"""

results = read_query(connection, query)
if results:
    print("first_name | last_name | birth_dete | gender | Salaries")
    print("-" * 80)
    for result in results:
        print(result)
else:
    print("No results found.")

openGauss Database connection successful
first_name | last_name | birth_dete | gender | Salaries
--------------------------------------------------------------------------------
('Udi', 'Zastre', datetime.datetime(1954, 7, 14, 0, 0), 'F', [{'from_date': '2001-12-23 00:00:00', 'to_date': '9999-01-01 00:00:00', 'salary': 61163}, {'from_date': '2000-12-23 00:00:00', 'to_date': '2001-12-23 00:00:00', 'salary': 60720}, {'from_date': '1999-12-24 00:00:00', 'to_date': '2000-12-23 00:00:00', 'salary': 59393}, {'from_date': '1998-12-24 00:00:00', 'to_date': '1999-12-24 00:00:00', 'salary': 55066}, {'from_date': '1997-12-24 00:00:00', 'to_date': '1998-12-24 00:00:00', 'salary': 52872}, {'from_date': '1996-12-24 00:00:00', 'to_date': '1997-12-24 00:00:00', 'salary': 52908}, {'from_date': '1995-12-25 00:00:00', 'to_date': '1996-12-24 00:00:00', 'salary': 52538}, {'from_date': '1994-12-25 00:00:00', 'to_date': '1995-12-25 00:00:00', 'salary': 49777}])


In [82]:
#5) 创建并执行一个函数 FUNC2，基于 FUNC1，通过该函数可以根据输入参数员工编号查询出该员工具体信息：包括名字、生日、性别、在不同时间段下的工资的均值。（函数嵌套）
connection = create_db_connection("localhost", "5432", "myuser", "123456abc.", "mydb")

query = """
CREATE OR REPLACE FUNCTION kk.FUNC1(
    p_emp_no INTEGER
)
RETURNS TABLE (
    first_name VARCHAR(50),
    last_name VARCHAR(50),
    birth_date DATE,
    gender VARCHAR(6),
    salaries JSON
) 
AS $$
BEGIN
    RETURN QUERY
    SELECT 
        e.first_name,
        e.last_name,
        e.birth_date,
        e.gender,
        (SELECT json_agg(
            json_build_object(
                'from_date', s.from_date,
                'to_date', s.to_date,
                'salary', s.salary
            )
            ORDER BY s.from_date DESC
        )
        FROM kk.salaries s
        WHERE s.emp_no = e.emp_no) AS salaries
    FROM 
        kk.employees e
    WHERE 
        e.emp_no = p_emp_no;
END;
$$ LANGUAGE PLPGSQL;

CREATE OR REPLACE FUNCTION kk.FUNC2(
    p_emp_no INTEGER
)
RETURNS TABLE (
    first_name VARCHAR(50),
    last_name VARCHAR(50),
    birth_date DATE,
    gender VARCHAR(6),
    avg_salary NUMERIC(10,2)
) 
AS $$
DECLARE
    salary_records JSON;
BEGIN
    SELECT f.salaries INTO salary_records
    FROM kk.FUNC1(p_emp_no) f;
    
    RETURN QUERY
    SELECT 
        f.first_name,
        f.last_name,
        f.birth_date,
        f.gender,
        (SELECT AVG((s->>'salary')::NUMERIC) 
         FROM json_array_elements(salary_records) s) AS avg_salary
    FROM 
        kk.FUNC1(p_emp_no) f
    LIMIT 1;
END;
$$ LANGUAGE PLPGSQL;

SELECT * FROM kk.FUNC2(11111);
"""

results = read_query(connection, query)
if results:
    print("first_name | last_name | birth_dete | gender | Salaries")
    print("-" * 80)
    for result in results:
        print(result)
else:
    print("No results found.")

openGauss Database connection successful
first_name | last_name | birth_dete | gender | Salaries
--------------------------------------------------------------------------------
('Udi', 'Zastre', datetime.datetime(1954, 7, 14, 0, 0), 'F', Decimal('55554.625000000000'))


In [83]:
#1) 创建一个触发器，实现当删除 employees 表的一条员工记录时，将该数据也同步删除在 dept_manager、dept_emp、salaries、titles 表中的记录。
connection = create_db_connection("localhost", "5432", "myuser", "123456abc.", "mydb")

query = """
DROP TRIGGER IF EXISTS TRI_EMP_DELETE ON kk.departments;
CREATE OR REPLACE FUNCTION kk.tri_emp_delete_func()
RETURNS TRIGGER
AS $$
BEGIN
    DELETE FROM kk.dept_manager 
    WHERE emp_no = OLD.emp_no;
    DELETE FROM kk.dept_emp 
    WHERE emp_no = OLD.emp_no;
    DELETE FROM kk.salaries 
    WHERE emp_no = OLD.emp_no;
    DELETE FROM kk.titles 
    WHERE emp_no = OLD.emp_no;
    RETURN OLD;
END;
$$ LANGUAGE plpgsql;

CREATE TRIGGER TRI_EMP_DELETE
BEFORE DELETE ON kk.employees
FOR EACH ROW 
EXECUTE PROCEDURE kk.tri_emp_delete_func();
"""

execute_query(connection, query)
results = read_query(connection, "SELECT * FROM kk.salaries WHERE emp_no = 10002;")
for result in results:
    print(result)
print("-" * 80)
execute_query(connection, "DELETE FROM kk.employees WHERE emp_no = 10002;")
print("-" * 80)
results = read_query(connection, "SELECT * FROM kk.salaries WHERE emp_no = 10002;")
for result in results:
    print(result)

openGauss Database connection successful
Query executed successfully
(10002, 65828, datetime.datetime(1996, 8, 3, 0, 0), datetime.datetime(1997, 8, 3, 0, 0))
(10002, 65909, datetime.datetime(1997, 8, 3, 0, 0), datetime.datetime(1998, 8, 3, 0, 0))
(10002, 67534, datetime.datetime(1998, 8, 3, 0, 0), datetime.datetime(1999, 8, 3, 0, 0))
(10002, 69366, datetime.datetime(1999, 8, 3, 0, 0), datetime.datetime(2000, 8, 2, 0, 0))
(10002, 71963, datetime.datetime(2000, 8, 2, 0, 0), datetime.datetime(2001, 8, 2, 0, 0))
(10002, 72527, datetime.datetime(2001, 8, 2, 0, 0), datetime.datetime(9999, 1, 1, 0, 0))
--------------------------------------------------------------------------------
Query executed successfully
--------------------------------------------------------------------------------


In [87]:
#3)验证触发器是否正常工作：分别执行以下 A,B 两种操作，验证 INSERT_OR_ UPDATE_SAL 触发器是否被触发?工作是否正确?如果正确，请观察 salaries 表
#中数据的变化是否与预期一致。插 入 两 条 新 数 据 
#(10005,80001,'1995-06-24 00:00:00','1996-06-24 00:00:00') ,
#(10006,78888 ,'2000-06-22 00:00:00','2002-09-22 00:00:00');
#更新数据:将 id 为 10001 的员工在所有时期的工资改为 90000。

connection = create_db_connection("localhost", "5432", "myuser", "123456abc.", "mydb")

query = """
INSERT INTO kk.salaries VALUES
(10005, 80001, '1995-06-24', '1996-06-24'),
(10006, 78888, '2000-06-22', '2002-09-22');
"""

execute_query(connection, query)
execute_query(connection, "UPDATE kk.salaries SET salary = 90000 WHERE emp_no = 10002;")

query = """
SELECT * FROM kk.salaries 
WHERE emp_no IN (10005, 10006, 10001)
ORDER BY emp_no, from_date;
"""
results = read_query(connection, query)
for result in results:
    print(result)

openGauss Database connection successful
Query executed successfully
Query executed successfully
(10001, 60117, datetime.datetime(1986, 6, 26, 0, 0), datetime.datetime(1987, 6, 26, 0, 0))
(10001, 62102, datetime.datetime(1987, 6, 26, 0, 0), datetime.datetime(1988, 6, 25, 0, 0))
(10001, 66074, datetime.datetime(1988, 6, 25, 0, 0), datetime.datetime(1989, 6, 25, 0, 0))
(10001, 66596, datetime.datetime(1989, 6, 25, 0, 0), datetime.datetime(1990, 6, 25, 0, 0))
(10001, 66961, datetime.datetime(1990, 6, 25, 0, 0), datetime.datetime(1991, 6, 25, 0, 0))
(10001, 71046, datetime.datetime(1991, 6, 25, 0, 0), datetime.datetime(1992, 6, 24, 0, 0))
(10001, 74333, datetime.datetime(1992, 6, 24, 0, 0), datetime.datetime(1993, 6, 24, 0, 0))
(10001, 75286, datetime.datetime(1993, 6, 24, 0, 0), datetime.datetime(1994, 6, 24, 0, 0))
(10001, 75994, datetime.datetime(1994, 6, 24, 0, 0), datetime.datetime(1995, 6, 24, 0, 0))
(10001, 76884, datetime.datetime(1995, 6, 24, 0, 0), datetime.datetime(1996, 6, 23, 

In [88]:
#4) 删除触发器 INSERT_OR_UPDATE_SAL。
connection = create_db_connection("localhost", "5432", "myuser", "123456abc.", "mydb")

execute_query(connection, "DROP TRIGGER IF EXISTS INSERT_OR_UPDATE_SAL ON kk.salaries;")

openGauss Database connection successful
Query executed successfully
